In [7]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv, dotenv_values

load_dotenv(override=True)
env_dict = dotenv_values(".env")
print("--- Contents of .env file ---")
for key, value in env_dict.items():
    # Masking the value so you don't accidentally print your full API key to your screen!
    masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
    print(f"{key}: {masked_value}")

--- Contents of .env file ---
OPENAI_API_KEY: sk-pr...-PAA
GOOGLE_API_KEY: AQ.Ab...dTQA
TAVILY_API_KEY: tvly-...keLk
USER_AGENT: rag-f.../1.0
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https....com
LANGSMITH_API_KEY: lsv2_...2276
LANGSMITH_PROJECT: langc...demy
HF_TOKEN: hf_vG...sddc
LANGCHAIN_TRACING_V2: true
LANGCHAIN_API_KEY: lsv2_...13cd
LANGCHAIN_PROJECT: langc...demo


In [8]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
    )
),
)
blog_docs = loader.load()

#split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=50
)

#make splits
splits = text_splitter.split_documents(blog_docs)

#index
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma 

# vectorstore = Chroma.from_documents(
#     documents=splits,
#     embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# 3. Create your Chroma vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=hf_embeddings
)
retriever = vectorstore.as_retriever()



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

# 2. Build the chain (Swapped ChatOpenAI for ChatGoogleGenerativeAI)

local_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

generate_queries = (
    prompt_perspectives 
    | local_llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# 4. Run it!
questions = generate_queries.invoke({"question": "What is an LLM agent?"})
print(questions)

['Explain the concept of an LLM agent in detail.', 'How do Large Language Model agents function or operate?', 'What are some real-world use cases or applications for implementing LLM agents?', 'Describe the technical architecture and core components (such as planning, memory, or tool usage) that define an LLM agent.', 'What capabilities allow an AI system to autonomously complete complex, multi-step tasks?']


In [9]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

#retrieve 
question = "what is task decomposition for llm agents"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


8

In [11]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


''

In [12]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

NameError: name 'prompt_rag_fusion' is not defined

In [17]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/2443664460.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


11

In [5]:
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

NameError: name 'retrieval_chain_rag_fusion' is not defined

In [ ]:
##Decomposition 

In [1]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model") # LM Studio automatically uses whatever model is loaded in the server
    
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

In [12]:
questions

['1. What is the role of a planning module in an LLM-powered autonomous agent system? (Focuses on decision-making and control flow)',
 '2. How do LLMs integrate memory components (e.g., vector databases, short-term/long-term memory) to maintain state within an autonomous agent? (Focuses on context management)',
 '3. What are the technical requirements for enabling tool use and external API interactions within a large language model agent framework? (Focuses on action capability and system integration)']

In [13]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [14]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """ Format Q and A pair """
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

llm = ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model") 

q_a_pairs = ""
for q in questions:
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

In [ ]:
answer

'The ability for an LLM agent to use external tools and APIs transforms it from a mere text predictor into an autonomous, action-capable system. The technical requirements span three main layers: **The Prompt Interface (Input)**, **The Runtime Backend (Execution)**, and **The Control Flow Logic (Orchestration)**.\n\nHere is a detailed breakdown of the technical requirements for enabling robust tool use and external API interactions.\n\n---\n\n### 🛠️ I. The Architectural Requirement: Implementing the ReAct Loop\n\nThe foundational requirement is establishing a rigid, structured operational cycle that manages state through actions and observations. This pattern is known as **ReAct (Reasoning + Action)**.\n\n**Technical Mechanism:**\n1.  **Thought ($\\text{LLM}$):** The LLM generates an internal reasoning step: *What* needs to be done and *Why*.\n2.  **Action ($\\text{LLM}$ $\\rightarrow$ System):** Based on the thought, the LLM must generate a structured output specifying which function/

In [20]:
from langchain.hub import hub
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt_rag = hub.pull("rlm/rag-prompt")

ModuleNotFoundError: No module named 'langchain.hub'